In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/cleaned-atkinson/atkinson_main_unique.csv
/kaggle/input/datasets/raesiow/cleaned-off/off_filtered_all_countries.csv


In [2]:
off_path ='/kaggle/input/datasets/raesiow/cleaned-off/off_filtered_all_countries.csv'
atk_path = '/kaggle/input/datasets/raesiow/cleaned-atkinson/atkinson_main_unique.csv'

In [3]:
off = pd.read_csv(off_path, dtype={"code": str})
# load the Atkinson GI table
atk = pd.read_csv(atk_path)

# rename Atkinson columns to the clearer names we agreed on
atk = atk.rename(columns={"carb_g": "avail_carb_per_100g", "name_core": "name_clean"})

# show rows and columns of each table
print("OFF:", off.shape)
print("Atkinson:", atk.shape)
# show Atkinson column names so we can check the renames worked
print(atk.columns.tolist())

OFF: (376917, 18)
Atkinson: (3061, 8)
['name_clean', 'gi_median', 'gi_min', 'gi_max', 'n_rows', 'avail_carb_per_100g', 'example_item', 'any_table1']


In [4]:
import re                                                      # tool for finding and replacing text patterns

preparation_words = ["boiled", "steamed", "raw", "ripe", "cooked", "uncooked",
                     "microwaved", "homemade", "fresh", "ns", "heated",'peeled','unpeeled','sliced']  # added "heated"
cooking_phrases = [r"home-cooked",                             # new: must go before "cooked" is removed
                   r"cooking method",                          # new: e.g. "cooking method ns"
                   r"for \d+(\.\d+)? ?h\b", r"\d+(\.\d+)? ?h\b",  # new: hours, e.g. "for 1.5 h", "1h", "2 h"
                   r"for \d+(\.\d+)? ?min\w*",                         # e.g. "for 15 min", "for 23 minutes"
                   r"\d+(\.\d+)? ?min\w*",                             # e.g. "15 min" without "for"
                   r"in salt(ed)? water", r"in water",         # changed: covers "salt" and "salted"
                   r"in a? ?(electric )?rice cooker",                      # e.g. "in rice cooker", "in a rice cooker"
                   r"tested (in|on) .*",                   # new: study info, e.g. "tested in young adults" (removes to the end)
                   r"cut into [\d.\-–]+ ?(mm|cm) ?\w*",     # new: e.g. "cut into 25 mm slices", "cut into 2.5-3 cm cubes"
                   r"in a? ?microwave( oven)?",                # e.g. "in microwave oven"
                   r"with [\d.]+ ?g salt",                  # new: e.g. "with 2 g salt", "with 2.5 g salt"
                   r"at \d+ ?[o°]?c\b",                     # new: temperatures, e.g. "at 175oc", "at 4oc"
                   r"on (pre-?)?heated charcoal",             # new: e.g. "on heated charcoal", "on preheated charcoal"
                   r"tested .*"]                            # study info at the end, e.g. "tested in older adults


def make_matching_name(name):                                  # turn an Atkinson name into a shorter version for matching
    shorter = name                                             # start from the cleaned name
    for phrase in cooking_phrases:                             # go through each cooking phrase
        shorter = re.sub(phrase, " ", shorter)                 # replace it with a space
    for word in preparation_words:                             # go through each preparation word
        shorter = re.sub(rf"\b{word}\b", " ", shorter)         # replace it only as a whole word ("cooked" won't touch "uncooked")
    shorter = " ".join(shorter.split())                        # squash repeated spaces into one
    return shorter if shorter else name                        # if nothing is left, keep the original name

atk["name_for_matching"] = atk["name_clean"].apply(make_matching_name)  # add the new column

name_changed = atk["name_for_matching"] != atk["name_clean"]   # True where the name got shorter
print(name_changed.sum())                                      # how many names changed
print(atk.loc[name_changed, ["name_clean", "name_for_matching"]].sample(20, random_state=42).to_string())  # 20 before/after examples

865
                                                                                                                                                       name_clean                                                                                                                           name_for_matching
738                                                                                               desiree potato peeled quartered boiled 15 min in unsalted water                                                                                                  desiree potato quartered in unsalted water
2634                                                                sweet potato ipomoea batatas minda cultivar skin on roasted on pre-heated charcoal for 45 min                                                                                 sweet potato ipomoea batatas minda cultivar skin on roasted
2296  sago starch metroxylon spp. with the addition of 25% red bean phaseolus vulgaris flo

In [5]:
def make_head_name(original_name):                                  # build a short matching name from the ORIGINAL Atkinson name
    no_brackets = re.sub(r"\(.*?\)", " ", str(original_name))      # remove anything in brackets, e.g. "(Unico, Canada)"
    no_brackets = re.sub(r"[()]", " ", no_brackets)                # new: remove any leftover single bracket
    parts = no_brackets.split(",")                                  # split the name at each comma
    first_two = " ".join(parts[:2])                                 # keep the first two parts, joined with a space
    cleaned = first_two.lower().replace("'", "")                    # lowercase and remove apostrophes, like name_clean on the OFF side
    cleaned = " ".join(cleaned.split())                             # squash repeated spaces
    return make_matching_name(cleaned)                              # also remove preparation words like "boiled", "raw"

In [6]:
atk["name_head"] = atk["example_item"].apply(make_head_name)  # add the new column

examples = atk.sample(12, random_state=42)          # 12 random Atkinson foods
for original, head in zip(examples["example_item"], examples["name_head"]):  # go through each one
    print("original:", original)                               # the name as written in the paper
    print("head:    ", head)                                   # our short matching name
    print()    

# show 10 random foods: original name next to head name
print(atk[["example_item", "name_head"]].sample(10, random_state=1).to_string())
# how many different head names there are
print("Different head names:", atk["name_head"].nunique())
# how many foods ended up with an empty head name
print("Empty head names:", (atk["name_head"] == "").sum())

original: Original malt flavor, Up & Go™, soy milk, rice cereal liquid breakfast (Sanitarium Health Foods, Australia)
head:     original malt flavor up & go™

original: Barley, rolled
head:     barley rolled

original: Long Grain White Rice in 90 Seconds, microwaved (SunRice brand, Rice Growers Co-Op., Australia)
head:     long grain white rice in 90 seconds

original: Manioc (cassava), consumed with 25 g coconut sambol
head:     manioc consumed with 25 g coconut sambol

original: Muesli bread, made from packet mix in bread making machine (Con Agra Inc., USA)
head:     muesli bread made from packet mix in bread making machine

original: Arabinose Sugar (95% sucrose + 5% L-arabinose) (Healtang Biotech Co., Ltd, China)
head:     arabinose sugar

original: White bread, candeal-flour
head:     white bread candeal-flour

original: Wheat bread, sliced (Pan Bauletto Bianco)
head:     wheat bread

original: Chapatti with 8% ß-glucan
head:     chapatti with 8% ß-glucan

original: Maltitol and S

In [7]:
print(atk.columns.tolist())

['name_clean', 'gi_median', 'gi_min', 'gi_max', 'n_rows', 'avail_carb_per_100g', 'example_item', 'any_table1', 'name_for_matching', 'name_head']


In [8]:
atk_before_merge = atk.copy()                                  # keep the unmerged table, just in case

def make_merge_key(name):                                      # build a key that ignores word order
    return " ".join(sorted(name.split()))                      # "white rice" and "rice white" both become "rice white"

atk["merge_key"] = atk["name_head"].apply(make_merge_key)      # add the key column

atk = atk.groupby("merge_key").agg(                            # one row per key, combining the foods that share it
    name_head=("name_head", "first"),                          # keep one of the head names for matching
    example_item=("example_item", "first"),  
    gi_median=("gi_median", "median"),                         # GI label: median across the merged foods
    gi_min=("gi_min", "min"),                                  # lowest GI seen for this key
    gi_max=("gi_max", "max"),                                  # highest GI seen for this key
    avail_carb_per_100g=("avail_carb_per_100g", "median"),     # carbs: median (NaN values are ignored)
    n_foods_merged=("name_head", "size"),                      # how many Atkinson foods were merged into this row
).reset_index()                                                # turn merge_key back into a normal column and renumber rows 0, 1, 2, ...

print(len(atk_before_merge), "->", len(atk))                   # number of Atkinson foods before vs after merging
print(atk.sort_values("n_foods_merged", ascending=False).head(10)[["name_head","n_foods_merged", "gi_median", "gi_min", "gi_max"]].to_string())  # the most-merged foods

3061 -> 2590
                               name_head  n_foods_merged  gi_median  gi_min  gi_max
1579                  potato novel clone              16       56.5    38.0    82.0
2525                          rice white              15       73.0    34.0    96.0
1691                   ice cream low-fat              13       34.0    25.0    55.0
1291                         bread white              12       68.5    51.0    94.0
2263  milk rice pudding yoplait le rice®              10       49.0    41.0    54.0
1764                dates khalas variety              10       50.5    29.0    56.0
2551                     spaghetti white               8       48.5    39.0    65.0
1432                             carrots               8       37.0    23.0    92.0
2586                                 yam               8       53.5    35.0    82.0
1275                           bread rye               7       50.0    45.0    67.0


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer   # tool that turns text into numbers

off= off.reset_index(drop=True)             # number OFF rows 0, 1, 2, ... so positions line up later
atk = atk.reset_index(drop=True)         # same for Atkinson
name_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4)) 
all_names = pd.concat([off["name_clean"], atk["name_head"]])  # stack OFF and Atkinson names into one list
name_vectorizer.fit(all_names)                                                    # learn the pieces from both lists
off_name_vectors = name_vectorizer.transform(off["name_clean"])  # turn OFF names into numbers
atkinson_name_vectors = name_vectorizer.transform(atk["name_head"])      # turn Atkinson names into numbers

print(off_name_vectors.shape)                                  # (number of OFF products, number of different pieces)
print(atkinson_name_vectors.shape)    

(376917, 128416)
(2590, 128416)


In [10]:
import numpy as np                                            # maths tools for arrays
import time                                                   # to measure how long it takes

batch_size = 64                                               # how many Atkinson foods we compare at once
top_k = 50                                                    # how many candidates to keep per Atkinson food
candidate_rows = []                                           # empty list; we'll add (atkinson row, off row, similarity) here
start_time = time.time()                                      # remember when we started

for batch_start in range(0, atkinson_name_vectors.shape[0], batch_size):     # go through Atkinson 64 rows at a time
    batch_vectors = atkinson_name_vectors[batch_start:batch_start + batch_size]  # the numbers for this batch of Atkinson names
    similarity = (batch_vectors @ off_name_vectors.T).toarray()              # similarity of each batch name to every OFF name (0 to 1)
    top_positions = np.argpartition(-similarity, top_k, axis=1)[:, :top_k]   # positions of the 50 highest similarities in each row
    for row_in_batch, off_positions in enumerate(top_positions):            # go through each Atkinson food in this batch
        atkinson_row = batch_start + row_in_batch                           # its row number in atkinson_table
        for off_row in off_positions:                                       # go through its 50 candidates
            candidate_rows.append((atkinson_row, off_row, similarity[row_in_batch, off_row]))  # save the pair and its similarity

candidates = pd.DataFrame(candidate_rows, columns=["atkinson_row", "off_row", "tfidf_similarity"])  # turn the list into a table
print(candidates.shape)                                       # should be (3061 × 50 = 153050, 3)
print(round(time.time() - start_time), "seconds")             # how long it took
print(candidates.head()) 

(129500, 3)
22 seconds
   atkinson_row  off_row  tfidf_similarity
0             0   138784          0.619175
1             0    76494          0.619175
2             0   138799          0.619175
3             0    31194          0.619175
4             0   106250          0.619175


In [11]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.0 MB/s eta 0:00:0000:0100:01


In [12]:
from rapidfuzz import fuzz                                    # tool for comparing two pieces of text

candidates["atkinson_name"] = atk.loc[candidates["atkinson_row"], "name_head"].values  # look up each Atkinson name by its row number
candidates["off_name"] = off.loc[candidates["off_row"], "name_clean"].values  # look up each OFF name; empty text if missing

name_scores = []                                              # empty list for the scores
for atkinson_name, off_name in zip(candidates["atkinson_name"], candidates["off_name"]):  # go through every candidate pair
    sort_score = fuzz.token_sort_ratio(atkinson_name, off_name)                     # word-order-insensitive score (0-100)
    set_score = fuzz.token_set_ratio(atkinson_name, off_name)                       # extra-words-insensitive score (0-100)
    name_scores.append(0.5 * sort_score + 0.5 * set_score)                          # average of the two
candidates["name_score"] = name_scores                        # add the scores as a column

good_name_matches = candidates[candidates["name_score"] >= 85]  # keep only pairs scoring 85 or more
print(len(good_name_matches))                                 # how many pairs passed
print(good_name_matches["atkinson_row"].nunique())            # how many Atkinson foods have at least one passing match
print(good_name_matches.sample(15, random_state=42)[["atkinson_name", "off_name", "name_score"]].to_string())  # 15 random passing pairs to eyeball

10303
759
                           atkinson_name                       off_name  name_score
63424                white bread refined                    white bread   86.666667
128304                               thé                            thé  100.000000
114330                       muesli lite                         muesli   85.294118
117051                  rice white mayas                     white rice   88.461538
64355                white bread toasted                    white bread   86.666667
128658                   yoghurt vanilla        protein vanilla yoghurt   89.473684
117679                 milk semi-skimmed  fresh welsh milk semi-skimmed   86.956522
4656                        nutella & go                        nutella   86.842105
55006               yoghurt black cherry                   black cherry   87.500000
105556          premium long grain white          long grain white rice   85.465465
118398                            muesli                     bio m

In [13]:
candidates_with_carbs = good_name_matches.copy()              # copy, so we don't change the original table
candidates_with_carbs["atkinson_carb"] = atk.loc[candidates_with_carbs["atkinson_row"], "avail_carb_per_100g"].values  # Atkinson available carbs per 100 g
candidates_with_carbs["off_carb"] = off.loc[candidates_with_carbs["off_row"], "carb_excl_fiber_g"].values  # OFF carbs without fibre per 100 g
candidates_with_carbs["carb_ratio"] = candidates_with_carbs["off_carb"] / candidates_with_carbs["atkinson_carb"]  # OFF ÷ Atkinson: 1 = same, 2 = OFF has double

print(candidates_with_carbs["carb_ratio"].describe())        # summary of the ratios (pairs with missing carbs are left out)
print(candidates_with_carbs["atkinson_carb"].isna().sum())   # how many pairs have no Atkinson carb value

columns_to_show = ["atkinson_name", "off_name", "atkinson_carb", "off_carb", "carb_ratio"]  # the columns we want to look at
print(candidates_with_carbs.sort_values("carb_ratio").head(15)[columns_to_show].to_string())  # 15 pairs where OFF has much LESS carb
print(candidates_with_carbs.sort_values("carb_ratio").tail(15)[columns_to_show].to_string())  # 15 pairs where OFF has much MORE carb
print(candidates_with_carbs.dropna(subset=["carb_ratio"]).sort_values("carb_ratio").tail(15)[columns_to_show].to_string())  # 15 pairs where OFF has much MORE carb (missing ratios removed first)

count    8296.000000
mean        1.170221
std         1.285933
min         0.000000
25%         0.790500
50%         0.983899
75%         1.132628
max        30.525000
Name: carb_ratio, dtype: float64
2007
          atkinson_name                     off_name  atkinson_carb  off_carb  carb_ratio
128841       watermelon            propel watermelon      10.526736       0.0         0.0
128761        water yam                        water      17.421603       0.0         0.0
128762        water yam                        water      17.421603       0.0         0.0
128763        water yam                        water      17.421603       0.0         0.0
128764        water yam                        water      17.421603       0.0         0.0
128765        water yam                        water      17.421603       0.0         0.0
128766        water yam                        water      17.421603       0.0         0.0
128767        water yam                        water      17.421603       

In [14]:
ratio = candidates_with_carbs["carb_ratio"]                       # the carb ratio for each pair (NaN if a carb value is missing)
carb_ok = ratio.isna() | ((ratio >= 0.2) & (ratio <= 5))          # True if we can't check it, OR the carbs are within 5 times either way
passed_carb_check = candidates_with_carbs[carb_ok]                # keep only the pairs that passed

print(len(candidates_with_carbs), "->", len(passed_carb_check))   # pairs before vs after the carb check
print(passed_carb_check["atkinson_row"].nunique())                # Atkinson foods that still have at least one match

10303 -> 9776
748


In [15]:
matches = passed_carb_check.sort_values("name_score", ascending=False)  # best name scores first
matches = matches.drop_duplicates("off_row")                    # each OFF product keeps only its first (= best) Atkinson food
products_per_food = matches.groupby("atkinson_row")["off_row"].transform("size")  # for each row: how many products its Atkinson food has
matches["food_weight"] = 1 / products_per_food                  # so each Atkinson food counts once in total
matches = matches.reset_index(drop=True)                        # renumber rows 0, 1, 2, ...

print(len(atk))                                  # Atkinson foods after the merge
print(len(matches))                              # matched OFF products (rows)
print(matches["atkinson_row"].nunique())         # Atkinson foods that got at least one match
print(matches["atkinson_row"].max() < len(atk))  # should be True
print(matches["food_weight"].sum())              # should equal the number of matched foods (each food's weights add up to 1)

2590
6851
629
True
629.0


In [16]:
matches.to_csv("/kaggle/working/atkinson_off_fuzzy_matches.csv", index=False)  # the matched pairs (6,851 rows)
atk.to_csv("/kaggle/working/atkinson_merged.csv", index=False)                  # the merged Atkinson table (2,590 rows)
print(pd.read_csv("/kaggle/working/atkinson_off_fuzzy_matches.csv").shape)      # read it back to check it saved: should be (6851, ...)
print(pd.read_csv("/kaggle/working/atkinson_merged.csv").shape)                 # should be (2590, ...)

(6851, 10)
(2590, 8)


In [17]:
audit = matches.sample(50, random_state=42).reset_index(drop=True)   # 50 random matched pairs, renumbered 0-49
audit["off_full_name"] = off.loc[audit["off_row"], "product_name"].values   # the OFF product name as written (not cleaned)
audit["off_category"] = off.loc[audit["off_row"], "pnns_groups_2"].values   # OFF food category, helps spot wrong food types
if "example_item" in atk.columns:                                    # only if you kept the original name in the merge cell
    audit["atkinson_original"] = atk.loc[audit["atkinson_row"], "example_item"].values  # Atkinson name as written in the paper

for number, row in audit.iterrows():                                 # go through the 50 pairs one by one
    print(number + 1)                                                # pair number 1-50
    print("  Atkinson:", row["atkinson_name"], "|", row.get("atkinson_original", ""))  # Atkinson head name | original name
    print("  OFF:     ", row["off_full_name"], "|", row["off_category"])  # OFF product name | category
    print("  carbs:   ", row["atkinson_carb"], "vs", row["off_carb"], "  score:", round(row["name_score"]))  # carbs side by side + name score
    print()                                                          # empty line between pairs=

1
  Atkinson: orange juice | Orange juice
  OFF:      Orange Juice | unknown
  carbs:    8.871540099361248 vs 12.2575954824421   score: 100

2
  Atkinson: puffed crispbread | Puffed Crispbread (Westons, Australia)
  OFF:      Crispbread | Bread
  carbs:    70.4225352112676 vs 56.0   score: 87

3
  Atkinson: potato crisps | Potato crisps
  OFF:      Potato crisps, bbq | Potatoes
  carbs:    nan vs 57.14   score: 93

4
  Atkinson: kidney beans red | Kidney beans (Phaseolus vulgaris L.), red, soaked 20 min, boiled 70 min
  OFF:      Dark Red Kidney Beans | Legumes
  carbs:    nan vs 13.8461538461538   score: 93

5
  Atkinson: carrots diced | Carrots, diced, frozen (Talleys Group Ltd)
  OFF:      Sliced Carrots | Vegetables
  carbs:    nan vs 3.3   score: 89

6
  Atkinson: prince gout chocolat | Prince gout chocolat (LU, France)
  OFF:      Prince Lu goût chocolat | Biscuits and cakes
  carbs:    72.46376811594203 vs 71.0   score: 88

7
  Atkinson: rice vermicelli | Rice vermicelli (produc

Hand audit of the 50 pairs above: 45 correct (90%).

In [18]:
off_columns = ["code", "product_name", "ingredients_text", "protein_g", "fat_g", "carb_g",
               "sugars_g", "fiber_g", "sat_fat_g", "sodium_g", "carb_excl_fiber_g"]   # OFF columns the MLP (and DistilBERT later) might need

gi_table = off.loc[matches["off_row"], off_columns].reset_index(drop=True)   # one row per matched product, in the same order as matches
gi_table["merge_key"] = atk.loc[matches["atkinson_row"], "merge_key"].values  # which Atkinson food it belongs to (used for grouping)
gi_table["atkinson_name"] = matches["atkinson_name"].values                   # Atkinson head name, for reading
gi_table["gi"] = atk.loc[matches["atkinson_row"], "gi_median"].values        # the GI label we want to predict
gi_table["food_weight"] = matches["food_weight"].values                       # so each Atkinson food counts once

print(gi_table.shape)                                    # should be (6851, 15)
print(gi_table["merge_key"].nunique())                   # should be 629
print(gi_table.isna().sum().to_string())                 # how many missing values in each column
print(gi_table["gi"].describe())                         # GI range, should be between 0 and ~110

(6851, 15)
629
code                 0
product_name         0
ingredients_text     0
protein_g            0
fat_g                0
carb_g               0
sugars_g             0
fiber_g              0
sat_fat_g            0
sodium_g             0
carb_excl_fiber_g    0
merge_key            0
atkinson_name        0
gi                   0
food_weight          0
count    6851.000000
mean       52.591957
std        16.763296
min         3.000000
25%        41.000000
50%        53.000000
75%        64.500000
max       120.000000
Name: gi, dtype: float64


In [20]:
print(gi_table.head())                                                                                                                                                              

            code product_name  \
0   853923002244      yoghurt   
1  4008471505006      Yoghurt   
2  4027394003161      Yoghurt   
3  6281007035040      Yoghurt   
4  5036589255550      Yoghurt   

                                    ingredients_text  protein_g      fat_g  \
0  Grade a pasteurized milk, cane sugar, honey, k...   3.086420   3.395062   
1   Joghurt, l.acidophilus, bifidobacterium, l.casei   4.900000   3.800000   
2  lait entier pasteurisé; crème (lait) pasteuris...   3.400000  10.000000   
3  Fresh Cow's Milk, Milk Solids, Culture, Stabil...   4.764706   1.117647   
4                                               milk   4.200000   3.800000   

     carb_g  sugars_g  fiber_g  sat_fat_g  sodium_g  carb_excl_fiber_g  \
0  7.098765  0.000000     0.00   2.160494   0.04321           7.098765   
1  5.600000  5.600000     0.00   2.600000   0.06800           5.600000   
2  5.400000  5.400000     0.11   6.700000   0.04400           5.400000   
3  6.411765  6.411765     0.00   0.8

In [21]:
from sklearn.model_selection import GroupKFold          # splits data so a group never appears in two folds

gi_table["fold"] = -1                                    # placeholder fold number
group_splitter = GroupKFold(n_splits=5)                  # 5 folds, grouped
for fold_number, (train_rows, test_rows) in enumerate(group_splitter.split(gi_table, groups=gi_table["merge_key"])):  # each loop gives one test fold
    gi_table.loc[test_rows, "fold"] = fold_number        # mark which fold these test rows belong to

print(gi_table.groupby("fold")["merge_key"].nunique())   # Atkinson foods per fold, roughly 126 each
print(gi_table.groupby("merge_key")["fold"].nunique().max())  # must be 1: every food sits in only ONE fold
gi_table.to_csv("/kaggle/working/gi_table_folds.csv", index=False)  # save for the MLP notebook

fold
0    126
1    125
2    126
3    126
4    126
Name: merge_key, dtype: int64
1


In [22]:
print((gi_table["fiber_g"] == 0).mean())   # share of products with exactly 0 g fibre

0.13253539629251204
